In [ ]:
import numpy as np
import pandas as pd
credits = pd.read_csv(r"C:\Users\91902\Desktop\projects\Movie recomendation\tmdb_5000_credits.csv")
movies = pd.read_csv(r"C:\Users\91902\Desktop\projects\Movie recomendation\tmdb_5000_movies.csv")



In [ ]:
movies.head(1)

In [ ]:
movies.shape

In [ ]:
credits.head(1)

In [ ]:
credits.shape

In [ ]:
movies = movies.merge(credits,on='title')#title,id(2 options we have)

In [ ]:
print(movies.columns)
# I choose columns on the basis which will work for content based filtering.
# genres,id,keywords,overview,cast(top3 ),crew(director)


In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres
# dictionary 

In [ ]:
import ast
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L 

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies.head()

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

In [ ]:


def convert3(text):
    L = []
    counter = 0
    
    # Check if text is already a list
    if isinstance(text, list):
        items = text
    else:
        items = ast.literal_eval(text)
    
    for i in items:
        if counter < 3:
            L.append(i['name'] if isinstance(i, dict) and 'name' in i else i)
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert3)
movies.head()


In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L 

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:

movies.sample(5)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else x) # string to list
movies.head()

In [ ]:
#Sneha goyal->SnehaGoyal
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies.head()

In [ ]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])


In [ ]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))#list to string
new.head()

In [ ]:
new['tags']=new['tags'].apply(lambda x:x.lower())
new.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer #5000 is my choice
cv = CountVectorizer(max_features=5000,stop_words='english') 
    

In [ ]:
vector = cv.fit_transform(new['tags']).toarray()

In [ ]:
vector.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
similarity

In [ ]:
def recommend(movie):
    index = new[new['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:11]:
        print(new.iloc[i[0]].title)
        
    

In [ ]:
recommend('Gandhi')

In [ ]:
recommend('Aladdin')

In [74]:
import  pickle

In [75]:
import pickle
pickle.dump(new,open('movie_list.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))